# DecisionTree — Point Cloud Segmentation

Beginner-friendly notebook. Runs top to bottom.

In [ ]:
# ── Install (run once if needed) ─────────────────────────────────────────────
# pip install torch laspy open3d numpy scikit-learn joblib tqdm mlflow matplotlib
# For PTv1/PTv2/GNN also:
# pip install torch-scatter torch-cluster -f https://data.pyg.org/whl/torch-<VER>+<CUDA>.html

import os, glob, copy, random, logging
import numpy as np
import torch
import torch.nn as nn
import open3d as o3d
import laspy
import joblib
import mlflow
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

# Use GPU if available, otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Running on: {DEVICE}")


In [ ]:
# ── Configuration — change values here, nowhere else ────────────────────────
CONFIG = {
    "train_dir"      : "data/train",      # labelled .las files
    "test_dir"       : "data/test",       # unlabelled .las files for inference
    "checkpoint_dir" : "checkpoints",     # where the best model is saved
    "num_classes"    : 2,                 # 0 = environment, 1 = wood powder
    "target_class"   : 1,                 # class we want to highlight (green)
    "val_ratio"      : 0.15,
    "test_ratio"     : 0.15,
    "seed"           : 42,
    # ── training ──
    "epochs"         : 100,
    "patience"       : 30,               # early stop if val mIoU doesn't improve
    "batch_size"     : 8,
    "num_points"     : 4096,             # points per training chunk
    "chunks_per_cloud": 4,
    "lr"             : 1e-3,
    "in_channels"    : 7,                # xyz + height + normals
    # ── visualization ──
    "max_vis_files"  : 3,                # how many test files to show in 3D
    # ── MLflow ──
    "mlflow_uri"     : "http://localhost:5000",
    "mlflow_experiment": "PointCloud_Segmentation",
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
NUM_CLASSES = CONFIG["num_classes"]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])


In [ ]:
# ── Point cloud loader ───────────────────────────────────────────────────────
def load_pointcloud(path):
    """Read a .las/.laz file. Returns (points[N,3], labels[N] or None)."""
    las = laspy.read(path)
    pts = np.column_stack([np.asarray(las.x),
                           np.asarray(las.y),
                           np.asarray(las.z)]).astype(np.float64)
    labels = None
    for key in ("classification", "label", "labels", "class"):
        if key in las.point_format.dimension_names:
            labels = np.asarray(getattr(las, key), dtype=np.int64)
            break
    return pts, labels

def list_files(folder):
    """Return sorted list of .las/.laz files in a folder."""
    files = []
    for ext in (".las", ".laz"):
        files += glob.glob(os.path.join(folder, "*" + ext))
    return sorted(files)

# ── Split labelled files into train / val / test ─────────────────────────────
all_files = list_files(CONFIG["train_dir"])
assert all_files, f"No .las files found in {CONFIG['train_dir']}"

rng   = np.random.RandomState(CONFIG["seed"])
order = rng.permutation(len(all_files))
n_val  = max(1, int(len(all_files) * CONFIG["val_ratio"]))
n_test = max(1, int(len(all_files) * CONFIG["test_ratio"]))

VAL_FILES   = [all_files[i] for i in order[:n_val]]
TEST_FILES  = [all_files[i] for i in order[n_val : n_val + n_test]]
TRAIN_FILES = [all_files[i] for i in order[n_val + n_test :]]
INFER_FILES = list_files(CONFIG["test_dir"])   # no labels, final inference only

log.info(f"train={len(TRAIN_FILES)}  val={len(VAL_FILES)}  "
         f"test={len(TEST_FILES)}  inference={len(INFER_FILES)}")


In [ ]:
# ── Handcrafted 8-channel features (for Classical ML) ────────────────────────
def point_features_ml(points, k=16):
    """8 geometric features per point: height, normal, linearity, planarity,
    sphericity, curvature, density, height-range."""
    from sklearn.neighbors import NearestNeighbors

    z     = points[:, 2]
    z_rel = (z - z.min()) / max(z.max() - z.min(), 1e-9)

    nbrs = NearestNeighbors(n_neighbors=k, algorithm="kd_tree",
                            n_jobs=-1).fit(points)
    _, idx = nbrs.kneighbors(points)

    nb   = points[idx]
    mu   = nb.mean(1, keepdims=True)
    cov  = np.einsum("nki,nkj->nij", nb - mu, nb - mu) / k
    w, V = np.linalg.eigh(cov)
    l3, l2, l1 = w[:,0], w[:,1], w[:,2]
    s = np.clip(l1, 1e-12, None)

    linearity  = (l1 - l2) / s
    planarity  = (l2 - l3) / s
    sphericity =  l3 / s
    curvature  =  l3 / np.clip(l1+l2+l3, 1e-12, None)
    nz_abs     = np.abs(V[:,:,0][:,2])
    r_k        = np.linalg.norm(nb[:,-1,:] - points, axis=1)
    density    = k / np.clip((4/3)*np.pi*r_k**3, 1e-9, None)
    h_range    = nb[:,:,2].max(1) - nb[:,:,2].min(1)

    X = np.column_stack([z_rel, nz_abs, linearity, planarity,
                         sphericity, curvature, np.log1p(density), h_range])
    return np.nan_to_num(X).astype(np.float32)

# ── Build train/val matrices ──────────────────────────────────────────────────
PER_FILE = 3000   # points sampled per file for training

log.info("Building ML feature matrices...")
X_train, y_train, X_val, y_val = [], [], [], []

for path in tqdm(TRAIN_FILES, desc="train features"):
    pts, lbl = load_pointcloud(path)
    if lbl is None:
        continue
    lbl = np.clip(lbl, 0, NUM_CLASSES - 1)
    idx = np.random.choice(len(pts), min(PER_FILE, len(pts)), replace=False)
    X_train.append(point_features_ml(pts[idx]))
    y_train.append(lbl[idx])

for path in tqdm(VAL_FILES, desc="val features"):
    pts, lbl = load_pointcloud(path)
    if lbl is None:
        continue
    lbl = np.clip(lbl, 0, NUM_CLASSES - 1)
    idx = np.random.choice(len(pts), min(PER_FILE, len(pts)), replace=False)
    X_val.append(point_features_ml(pts[idx]))
    y_val.append(lbl[idx])

X_train = np.vstack(X_train);  y_train = np.concatenate(y_train)
X_val   = np.vstack(X_val);    y_val   = np.concatenate(y_val)

# Normalise features
scaler  = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
log.info(f"Train: {X_train.shape}  Val: {X_val.shape}")


In [ ]:
# ── Metrics ──────────────────────────────────────────────────────────────────
def compute_miou(true_labels, pred_labels, num_classes):
    """Compute mean Intersection-over-Union across all classes."""
    ious = []
    for c in range(num_classes):
        tp = int(((true_labels == c) & (pred_labels == c)).sum())
        fp = int(((true_labels != c) & (pred_labels == c)).sum())
        fn = int(((true_labels == c) & (pred_labels != c)).sum())
        if tp + fp + fn > 0:
            ious.append(tp / (tp + fp + fn))
    return float(np.mean(ious)) if ious else 0.0


In [ ]:
# ── MLflow setup ─────────────────────────────────────────────────────────────
try:
    mlflow.set_tracking_uri(CONFIG["mlflow_uri"])
    mlflow.set_experiment(CONFIG["mlflow_experiment"])
    MLFLOW_OK = True
    log.info(f"MLflow tracking: {CONFIG['mlflow_uri']}")
except Exception as e:
    MLFLOW_OK = False
    log.warning(f"MLflow not available ({e}) — training continues without logging")


## Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
MODEL_NAME='DecisionTree'
clf = DecisionTreeClassifier(random_state=CONFIG['seed'])

## Training

In [ ]:
# ── Training (fit the classifier) ────────────────────────────────────────────
ckpt_path = os.path.join(CONFIG["checkpoint_dir"], f"{MODEL_NAME}_best.joblib")

if os.path.exists(ckpt_path):
    log.info(f"Checkpoint found at {ckpt_path} — loading, skipping training.")
    blob    = joblib.load(ckpt_path)
    clf     = blob["model"]
    scaler  = blob["scaler"]
else:
    log.info(f"Training {MODEL_NAME}...")
    run_name = f"{MODEL_NAME}_seg"
    with (mlflow.start_run(run_name=run_name) if MLFLOW_OK
          else open(os.devnull, "w")) as _:

        clf.fit(X_train, y_train)

        # validate
        val_preds = clf.predict(X_val)
        val_miou  = compute_miou(y_val, val_preds, NUM_CLASSES)
        log.info(f"Val mIoU: {val_miou:.4f}")

        if MLFLOW_OK:
            mlflow.log_params({"model": MODEL_NAME})
            mlflow.log_metric("val_miou", val_miou)

    joblib.dump({"model": clf, "scaler": scaler}, ckpt_path)
    log.info(f"Saved → {ckpt_path}")


## Visualization helper

In [ ]:
# ── Visualization: Open3D ────────────────────────────────────────────────────
# Green = target (wood powder)   |   Red = others (environment)
def visualize_segmentation(points, predictions, title="Segmentation"):
    """Open an Open3D window showing the segmentation result."""
    colors = np.zeros((len(points), 3), dtype=np.float64)
    colors[predictions == CONFIG["target_class"]] = [0.0, 0.8, 0.0]   # green
    colors[predictions != CONFIG["target_class"]] = [0.8, 0.0, 0.0]   # red

    pcd = o3d.geometry.PointCloud()
    # centre the cloud so it appears at the origin
    center = points.mean(axis=0)
    pcd.points = o3d.utility.Vector3dVector(
        (points - center).astype(np.float64))
    pcd.colors = o3d.utility.Vector3dVector(colors)

    # XYZ axes
    span = float((points.max(0) - points.min(0)).max()) * 0.5
    axes = o3d.geometry.TriangleMesh.create_coordinate_frame(size=span)

    n_target = int((predictions == CONFIG["target_class"]).sum())
    n_other  = len(predictions) - n_target
    full_title = (f"{title}  |  green(target)={n_target:,}  "
                  f"red(others)={n_other:,}")
    o3d.visualization.draw_geometries([pcd, axes],
                                       window_name=full_title,
                                       width=1280, height=800)


## Inference helper

In [ ]:
# ── Full-cloud inference for Classical ML ────────────────────────────────────
def predict_full_cloud_ml(path, batch=80_000):
    """Predict every point in a file using the trained sklearn model."""
    pts, lbl = load_pointcloud(path)
    lbl = np.zeros(len(pts), dtype=np.int64) if lbl is None else lbl
    n   = len(pts)
    preds = np.empty(n, np.int64)
    for s in range(0, n, batch):
        e    = min(s + batch, n)
        feat = point_features_ml(pts[s:e])
        feat = scaler.transform(feat)
        preds[s:e] = clf.predict(feat)
    return pts, preds, lbl


## Testing  (held-out test files with labels)

In [ ]:
# ── Testing on held-out test files ───────────────────────────────────────────
log.info("=== TESTING ===")
test_mious = []
for path in TEST_FILES:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, lbl = predict_full_cloud_ml(path)
    miou = compute_miou(lbl, preds, NUM_CLASSES)
    test_mious.append(miou)
    log.info(f"  {name}: mIoU = {miou:.4f}")

log.info(f"Mean test mIoU: {np.mean(test_mious):.4f}")

# ── Visualize the first few test files ───────────────────────────────────────
for path in TEST_FILES[:CONFIG["max_vis_files"]]:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, _ = predict_full_cloud_ml(path)
    visualize_segmentation(pts, preds, title=f"{MODEL_NAME} | {name}")


## Final Inference  (`data/test`, no labels)

In [ ]:
# ── Final inference on data/test (no labels needed) ──────────────────────────
if INFER_FILES:
    log.info("=== FINAL INFERENCE ===")
    for path in INFER_FILES:
        name = os.path.splitext(os.path.basename(path))[0]
        pts, preds, _ = predict_full_cloud_ml(path)
        n_tgt = int((preds == CONFIG["target_class"]).sum())
        log.info(f"  {name}: target={n_tgt:,} / total={len(preds):,}")
        visualize_segmentation(pts, preds, title=f"INFERENCE | {MODEL_NAME} | {name}")
else:
    log.info("data/test is empty — skipping inference.")
